# Weather → Outfit Suggestions


Ви зробите проєктик, який:
* отримує прогноз погоди на завтра через Open-Meteo API (без API-ключа)
* витягує з прогнозу ключові показники (температура, опади, вітер, хмарність тощо)
* формує промпт і надсилає його в Gemini API (чи іншу LLM, якщо вам цікаво)
* отримує відповідь строго у JSON, щоб потім можна було легко пройтися циклом по варіантах одягу та показати їх у зручному вигляді.


Поради по API:

* Для погоди: [Open-Meteo](https://open-meteo.com/en/docs) — зручно, бо не потребує ключа.

* Для LLM: у Python актуально використовувати Google Gen AI SDK (python-genai) або AI з google.colab.


Очікуваний результат: outfits.json - структура на кшталт (не обов'язково саме така, якщо маєте кращі ідеї):

```
{
  "city": "Kyiv",
  "date": "2026-01-22",
  "outfits": [
    {"title": "...", "items": ["...","..."], "notes": "..."},
    ...
  ]
}

```



# Завдання

Реалізуйте 5 функцій: HTTP-запит, парсинг прогнозу, побудова промпта, виклик Gemini, перевірка схеми відповіді.

In [ ]:
import requests
import json
def get_tomorrow_weather_openmeteo(
    lat: float = 50.45,
    lon: float = 30.52,
    timezone: str = "Europe/Kyiv",
) -> dict:
    """
    Отримує прогноз погоди на завтра з Open-Meteo за координатами.

    Вимоги:
    - Зробити HTTP GET запит до Open-Meteo Forecast API.
    - Запитати хоча б daily-поля:
        - temperature_2m_min
        - temperature_2m_max
        - precipitation_sum (або precipitation_probability_max)
        - windspeed_10m_max (або windgusts_10m_max)
      (Можна додати cloudcover або weathercode.)
    - Повернути "сирий" JSON (dict) відповіді API.
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
      "latitude": lat,
      "longitude": lon,
      "timezone": timezone,
      "daily": ["sunrise", "sunset", "snowfall_sum", "showers_sum", "temperature_2m_min", "temperature_2m_max", "precipitation_probability_max", "windgusts_10m_max"],
      "forecast_days": 2
    }

    resp = requests.get(url, params=params)
    resp.raise_for_status()
    return resp.json()
forecast_json = get_tomorrow_weather_openmeteo()
def save_to_json(summary, path):
  try:
    with open(path, 'w', encoding='utf-8') as f:
      json.dump(summary, f, indent=2, ensure_ascii=False)
  except Exception as e:
        print(f"Сталась непередбачувана помилка [{type(e).__name__}]: {str(e)}")
  return None
path = 'forecast.json'
save_to_json(forecast_json, path)


In [ ]:
def extract_weather_summary_for_prompt(forecast_json: dict) -> dict:
    """
    Перетворює відповідь Open-Meteo у компактний словник для промпта до Gemini.

    Вимоги:
    - Витягнути дані саме за "завтра".
      Найкраще знайти завтра за датою.
    - Повернути dict лише з простими значеннями (без великих масивів), напр.:
        {
          "date": "YYYY-MM-DD",
          "temp_min_c": float,
          "temp_max_c": float,
          "precip_mm": float,
          "wind_max_ms": float,
          "summary": str
        }
    """
    daily_units = forecast_json.get('daily_units', None)
    daily = forecast_json.get('daily', None)
    return {key: [f"{v[v.find(':')-2:] if isinstance(v, str) and ':' in v else v} {daily_units[key].replace('iso8601','')}".strip() for v in val][1] for key, val in daily.items()} if daily and daily_units else {}


weather_summary = extract_weather_summary_for_prompt(forecast_json)

#print(weather_summary)


In [ ]:
print(weather_summary)


{'time': '2026-03-27', 'sunrise': '05:45', 'sunset': '18:21', 'snowfall_sum': '0.0 cm', 'showers_sum': '0.0 mm', 'temperature_2m_min': '6.8 °C', 'temperature_2m_max': '17.2 °C', 'precipitation_probability_max': '3 %', 'windgusts_10m_max': '18.4 km/h'}


In [ ]:
def build_gemini_outfit_prompt(
    weather_summary: dict,
    city: str
) -> str:
    """
    Будує промпт для Gemini так, щоб модель повернула ВАЛІДНИЙ JSON без зайвого тексту.

    Вимоги:
    - Явно написати: "Return JSON only" та "No markdown".
    - Описати схему відповіді (ключі та типи), наприклад:

      {
        "city": "<str>",
        "date": "<YYYY-MM-DD>",
        "outfits": [
          {
            "title": "<str>",
            "items": ["<str>", "<str>", ...],
            "notes": "<str>",
            "warnings": ["<str>", ...]
          }
        ]
      }
    """
    i_desc = ["Дата ", ", світанок о ", ", захід сонця о ",
              ", рівень снігових опадів за день ", ", рівень дощових опадів за день ",
              ", температура повітря від ", ", температура повітря до ",
              ", максимальна ймовірність опадів ", ", пориви вітру до "
              ]
    prompt_dict = {
        i_desc[i]: val for i, val in enumerate(weather_summary.values())
        if not ('опад' in i_desc[i] and  '0.0 ' in val)
    }
    prompt_string ="".join([f"{k}{v}" for k, v in prompt_dict.items()])
    prompt = f"""
              Використовуючи дані прогнозу погоди на завтра, надай будь ласка
              два-три варіанти outfits для середньостатистичного жителя
              міста {city}.

              Ось дані по прогнозу на завтра:
              {prompt_string}

              Це формат відповіді, який я очікую від тебе, ТІЛЬКИ JSON, БЕЗ блоків коду та БЕЗ markdown:

              {{
                "city": "<str>",
                "date": "<YYYY-MM-DD>",
                "outfits": [
                  {{
                    "title": "<str> (Назва або узагальнений опис списку речей)",
                    "items": ["<str>", "<str> (Перелік речей у списку)"],
                    "notes": "<str> (Пояснення, чому саме такий список потрібно вдягнути)",
                    "warnings": ["<str> (Особливі попередження, якщо потрібно на щось звернути увагу, наприклад  можлива ожеледиця, або потрібна парасолька)", ...]
                  }}
                ]
              }}

    """
    return prompt
gemini_prompt = build_gemini_outfit_prompt(weather_summary, 'Київ')

In [ ]:
import json
import time
from pydantic import json_schema

from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
def call_gemini_for_limits(
    api_key: str,
    prompt: str,
    model: str = "gemini-2.5-flash",
    timeout_s: int = 60
) -> dict:
    """
    Викликає Gemini API та повертає розпарсений JSON як dict.

    Вимоги:
    - Отримати текст відповіді.
    - Розпарсити JSON (json.loads).
    - Якщо відповідь містить зайвий текст навколо JSON:
        - спробувати витягнути JSON-блок від першої '{' до останньої '}'.

    Повертає:
    - dict зі структурою, яку ви описали у build_gemini_outfit_prompt().
    """
    prompt = ("Надай будь ласка у відповідь мій поточний ліміт на запити по обробці фото лічильників.
              "Потрібно обробити близько 130 фото, щоб отримати показники з кожного. Чи зможу я це зробити запитами на тебе і які мають бути таймаути?"
              "Якщо таке є, ще підкажи модель, яка може це зробити із більшим лімітом безкоштовно"
              "Замість крапок вкажи актуальні ліміти для мого поточного апі ключа"
              "Кількість моделей для відображення в результатах - всі доступні для мого поточного апі ключа"
              "Відповідь текстова Ліміт - ... запитів за хвилину на обробку фото, ... запитів за добу, ... запитів за місяць"
              "Окремі рядки формат:"
              "Доступна Модель 1 ... Ліміт - ... запитів за хвилину на обробку фото, ... запитів за добу, ... запитів за місяць"
              "Доступна Модель 2 ... Ліміт - ... запитів за хвилину на обробку фото, ... запитів за добу, ... запитів за місяць")
    if timeout_s>0:
      time.sleep(timeout_s)
    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(
        model=model,
        contents=prompt)
    return response.text



response = call_gemini_for_outfits(GEMINI_API_KEY,gemini_prompt,timeout_s=0)


SyntaxError: unterminated string literal (detected at line 27) (86296133.py, line 27)

In [ ]:
def format_outfit_response(data: dict) -> str:
    """
    Перевіряє, що відповідь Gemini має очікувану структуру.
    Форматує JSON у рядок, який може читати людина. Довільний формат.
    """
    try:
        source_dict = data
        if isinstance(source_dict, dict):
          city: str = source_dict.get('city')
          date: str = source_dict.get('date')
          outfits_list: list = source_dict.get('outfits')

          formatted_outfits =[]
          for i, v in enumerate(outfits_list, 1):
              title = v.get('title')
              items = ", ".join(v.get('items'))
              notes = v.get('notes')
              warnings = "\n-- ".join(v.get('warnings'))

              text = f"[{i}] {title}\n Речі: {items}\n !Порада!:\n {notes}\n \nБудьте уважні!\n{warnings}"
              formatted_outfits.append(text)

          all_text = "\n\n".join(formatted_outfits)

          answer = f"""
                Згідно прогнозу погоди на завтра ({date}) у місті {city} можу запропонувати {len(outfits_list)} різних підходи до Вашого одягу:

                {all_text}

                    """

          return answer
    except Exception as e:
        print(f"Сталась непередбачувана помилка [{type(e).__name__}]: {str(e)}")
    return None


print(format_outfit_response(response))


                Згідно прогнозу погоди на завтра (2026-03-27) у місті Київ можу запропонувати 3 різних підходи до Вашого одягу:

                [1] Практичний повсякденний образ
 Речі: Легкий светр або лонгслів, Джинсова куртка або легкий бомбер, Джинси або брюки чінос, Зручні кросівки або кеди
 !Порада!:
 Ідеально підходить для мінливої весняної погоди. Вранці куртка захистить від прохолоди (+6.8 °C), а вдень її можна буде зняти, якщо потеплішає до +17.2 °C.
 
Будьте уважні!


[2] Офісний кежуал з елементами тепла
 Речі: Сорочка або блуза, Кардиган або легкий блейзер, Класичні брюки або темні джинси, Лофери або черевики на низькому ходу
 !Порада!:
 Цей образ поєднує діловий стиль зі зручністю. Кардиган або блейзер додадуть тепла вранці, а після обіду їх можна буде розстебнути або зняти, враховуючи приємну денну температуру.
 
Будьте уважні!


[3] Комфортний образ для прогулянок
 Речі: Футболка або легкий гольф, Флісова кофта або світшот, Легка демісезонна куртка (наприклад, вітровка

In [ ]:
import time

from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
def call_gemini_for_outfits(
    api_key: str,
    prompt: str,
    model: str = "gemini-2.5-flash",
    timeout_s: int = 60
) -> dict:
    """
    Викликає Gemini API та повертає текст.

    Вимоги:
    - Отримати текст відповіді.
    - Розпарсити JSON (json.loads).
    - Якщо відповідь містить зайвий текст навколо JSON:
        - спробувати витягнути JSON-блок від першої '{' до останньої '}'.

    Повертає:
    - dict зі структурою, яку ви описали у build_gemini_outfit_prompt().
    """
    if timeout_s>0:
      time.sleep(timeout_s)
    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config={
            "response_mime_type": 'application/json',
        })
    try:
      return json.loads(response.text)
    except json.JSONDecodeError as e:
      print(f"Вибачте, не маю змоги декодувати текст у json.")
      print(f"При обробці рядка {e.lineno} у стовпчику {e.colno} сталась помилка:")
      print(f"{e.msg}")
      return response.text


response = call_gemini_for_outfits(GEMINI_API_KEY,gemini_prompt,timeout_s=0)